In [1]:
from pathlib import Path
from itertools import product
import copy
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

In [2]:
DATA_PATH = Path("train.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("hw1/train.csv")
data = pd.read_csv(DATA_PATH)
feature_names = [f"sensor_{i}" for i in range(6)]
X = data[feature_names].to_numpy(dtype=np.float32)
y = data["anomaly"].to_numpy(dtype=np.float32)

indices = np.arange(len(y))
train_idx, holdout_idx = train_test_split(
    indices, test_size=0.4, stratify=y, random_state=SEED,
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.5, stratify=y[holdout_idx], random_state=SEED,
)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X[train_idx]), dtype=torch.float32, device=DEVICE)
y_train = torch.tensor(y[train_idx], device=DEVICE)
X_val = torch.tensor(scaler.transform(X[val_idx]), dtype=torch.float32, device=DEVICE)
y_val = y[val_idx]
for name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(f"{name}: n={len(idx)}, anomalies={int(y[idx].sum())}, share={y[idx].mean():.3%}")

train: n=4696, anomalies=109, share=2.321%
validation: n=1566, anomalies=37, share=2.363%
test: n=1566, anomalies=36, share=2.299%


In [3]:
class AnomalyMLP(nn.Module):
    def __init__(self, input_dim=6, dropout=0.0, use_batchnorm=False,
                 hidden_layers=(64, 32), activation_fn="relu"):
        super().__init__()
        activations = {
            "relu": nn.ReLU, "leaky_relu": nn.LeakyReLU, "gelu": nn.GELU,
            "tanh": nn.Tanh, "sigmoid": nn.Sigmoid,
        }
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(activations[activation_fn]())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x).squeeze(-1)


def make_model(config):
    return AnomalyMLP(
        input_dim=X_train.shape[1],
        **{key: config[key] for key in
           ("dropout", "use_batchnorm", "hidden_layers", "activation_fn")},
    ).to(DEVICE)


def fit_model(config, X_fit, y_fit, validation=None):
    seed_everything()
    model = make_model(config)
    optimizer_class = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW,
                       "sgd": torch.optim.SGD}[config["optimizer"]]
    extra = {"momentum": 0.9} if config["optimizer"] == "sgd" else {}
    optimizer = optimizer_class(model.parameters(), lr=config["lr"],
                                weight_decay=config["weight_decay"], **extra)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([config["pos_weight"]], dtype=torch.float32, device=DEVICE)
    )
    generator = torch.Generator().manual_seed(SEED)
    best_score = -np.inf
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        order = torch.randperm(len(X_fit), generator=generator)
        batches = list(order.split(config["batch_size"]))
        if len(batches) > 1 and len(batches[-1]) == 1:
            batches[-2] = torch.cat((batches[-2], batches[-1]))
            batches.pop()
        for idx in batches:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_fit[idx]), y_fit[idx])
            loss.backward()
            optimizer.step()
        if validation is not None:
            model.eval()
            with torch.inference_mode():
                probabilities = model(validation[0]).sigmoid().cpu().numpy()
            score = float(average_precision_score(validation[1], probabilities))
            if score > best_score:
                best_score = score
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= config["patience"]:
                break
    if validation is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, {"val_ap": best_score, "best_epoch": best_epoch, "epochs_run": epoch}


def train_candidate(config):
    model, metrics = fit_model(config, X_train, y_train, validation=(X_val, y_val))
    return metrics, copy.deepcopy(model.state_dict())


results = []
trial_cache = {}
best = None

def search_stage(name, base_config, grid):
    global best
    keys = list(grid)
    combinations = list(product(*(grid[key] for key in keys)))
    stage_best = None
    for number, values in enumerate(combinations, 1):
        config = {**base_config, **dict(zip(keys, values))}
        cache_key = tuple(sorted(config.items()))
        if cache_key in trial_cache:
            metrics = trial_cache[cache_key]
            state = None
        else:
            metrics, state = train_candidate(config)
            trial_cache[cache_key] = metrics
        score = metrics["val_ap"]
        results.append({"stage": name, **config, **metrics})
        if stage_best is None or score > stage_best["val_ap"]:
            stage_best = {**metrics, "config": config.copy()}
        if best is None or score > best["val_ap"]:
            best = {**metrics, "config": config.copy(), "state_dict": state}
        if number % 10 == 0 or number == len(combinations):
            print(f"{name}: {number}/{len(combinations)}, best validation AP={stage_best["val_ap"]:.5f}", flush=True)
    print("Победитель этапа:", stage_best, flush=True)
    return stage_best["config"].copy()

In [4]:
base_config = dict(
    dropout=0.0, use_batchnorm=False, weight_decay=0.0,
    hidden_layers=(64, 32), activation_fn="relu", optimizer="adam",
    lr=1e-3, batch_size=128, epochs=200, patience=20, pos_weight=1.0,
)
architecture_grid = {
    "dropout": [0.0, 0.3],
    "use_batchnorm": [False, True],
    "weight_decay": [0.0, 1e-4],
    "hidden_layers": [(64, 32), (128, 64, 32)],
    "activation_fn": ["relu", "gelu", "tanh"],
}
best_architecture = search_stage("architecture", base_config, architecture_grid)

architecture: 10/48, best validation AP=0.74964
architecture: 20/48, best validation AP=0.76553
architecture: 30/48, best validation AP=0.76553
architecture: 40/48, best validation AP=0.76553
architecture: 48/48, best validation AP=0.76553
Победитель этапа: {'val_ap': 0.7655326532249473, 'best_epoch': 9, 'epochs_run': 29, 'config': {'dropout': 0.0, 'use_batchnorm': True, 'weight_decay': 0.0, 'hidden_layers': (128, 64, 32), 'activation_fn': 'relu', 'optimizer': 'adam', 'lr': 0.001, 'batch_size': 128, 'epochs': 200, 'patience': 20, 'pos_weight': 1.0}}


In [5]:
training_grid = {
    "optimizer": ["adam", "adamw", "sgd"],
    "lr": [1e-3, 1e-2, 5e-2],
    "batch_size": [64, 128],
}
best_training = search_stage("training", best_architecture, training_grid)

training: 10/18, best validation AP=0.78896
training: 18/18, best validation AP=0.79329
Победитель этапа: {'val_ap': 0.7932911619930373, 'best_epoch': 9, 'epochs_run': 29, 'config': {'dropout': 0.0, 'use_batchnorm': True, 'weight_decay': 0.0, 'hidden_layers': (128, 64, 32), 'activation_fn': 'relu', 'optimizer': 'sgd', 'lr': 0.05, 'batch_size': 128, 'epochs': 200, 'patience': 20, 'pos_weight': 1.0}}


In [6]:
positive_count = int(y_train.sum().item())
class_ratio = (len(y_train) - positive_count) / positive_count
weight_grid = {"pos_weight": sorted(set([1.0, float(np.sqrt(class_ratio)),
                                        float(class_ratio / 2), float(class_ratio)]))}
best_weighted = search_stage("pos_weight", best_training, weight_grid)
print("\nЛучшая конфигурация:", best["config"])
print(f"Validation AP: {best["val_ap"]:.5f}")
print(f"Лучшая эпоха: {best["best_epoch"]}, выполнено: {best["epochs_run"]}")
print("\nТоп-10 результатов (включая повторные конфигурации на границах этапов):")
for row in sorted(results, key=lambda row: row["val_ap"], reverse=True)[:10]:
    print(row)

pos_weight: 4/4, best validation AP=0.79329
Победитель этапа: {'val_ap': 0.7932911619930373, 'best_epoch': 9, 'epochs_run': 29, 'config': {'dropout': 0.0, 'use_batchnorm': True, 'weight_decay': 0.0, 'hidden_layers': (128, 64, 32), 'activation_fn': 'relu', 'optimizer': 'sgd', 'lr': 0.05, 'batch_size': 128, 'epochs': 200, 'patience': 20, 'pos_weight': 1.0}}

Лучшая конфигурация: {'dropout': 0.0, 'use_batchnorm': True, 'weight_decay': 0.0, 'hidden_layers': (128, 64, 32), 'activation_fn': 'relu', 'optimizer': 'sgd', 'lr': 0.05, 'batch_size': 128, 'epochs': 200, 'patience': 20, 'pos_weight': 1.0}
Validation AP: 0.79329
Лучшая эпоха: 9, выполнено: 29

Топ-10 результатов (включая повторные конфигурации на границах этапов):
{'stage': 'training', 'dropout': 0.0, 'use_batchnorm': True, 'weight_decay': 0.0, 'hidden_layers': (128, 64, 32), 'activation_fn': 'relu', 'optimizer': 'sgd', 'lr': 0.05, 'batch_size': 128, 'epochs': 200, 'patience': 20, 'pos_weight': 1.0, 'val_ap': 0.7932911619930373, 'bes

In [7]:
best_model = make_model(best["config"])
best_model.load_state_dict(best["state_dict"])
best_model.eval()
X_test = torch.tensor(scaler.transform(X[test_idx]), dtype=torch.float32, device=DEVICE)
y_test = y[test_idx]
with torch.inference_mode():
    test_probabilities = best_model(X_test).sigmoid().cpu().numpy()
print(f"Test AP: {average_precision_score(y_test, test_probabilities):.5f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probabilities):.5f}")

Test AP: 0.57963
Test ROC-AUC: 0.91777


In [8]:
final_config = best["config"].copy()
final_config["epochs"] = best["best_epoch"]
final_scaler = StandardScaler()
X_full = torch.tensor(final_scaler.fit_transform(X), dtype=torch.float32, device=DEVICE)
y_full = torch.tensor(y, dtype=torch.float32, device=DEVICE)
final_model, _ = fit_model(final_config, X_full, y_full)

test_data = pd.read_csv(DATA_PATH.parent / "test.csv")
submission = pd.read_csv(DATA_PATH.parent / "sample_submission.csv")
X_submission = torch.tensor(
    final_scaler.transform(test_data[feature_names].to_numpy(dtype=np.float32)),
    dtype=torch.float32, device=DEVICE,
)
with torch.inference_mode():
    submission_scores = final_model(X_submission).sigmoid().cpu().numpy()
score_by_id = pd.Series(submission_scores, index=test_data["id"])
submission["anomaly_score"] = submission["id"].map(score_by_id)
submission_path = DATA_PATH.parent / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Обучено на {len(X_full)} строках, эпох: {final_config["epochs"]}")
print(f"Сохранено: {submission_path}, строк: {len(submission)}")
print(submission.head().to_string(index=False))


Обучено на 7828 строках, эпох: 9
Сохранено: hw1/submission.csv, строк: 3355
    id  anomaly_score
109042       0.001486
103298       0.001340
106193       0.002373
106520       0.001276
105007       0.000183
